In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpg3cevotq".


In [ ]:
%%writefile matmul.cu

#include <iostream>          // Input Output
#include <cuda_runtime.h>   // CUDA functions
#include <limits>           // numeric_limits

using namespace std;


// ================= CUDA KERNEL =================

// GPU Kernel Function
__global__ void multiply(int* A, int* B, int* C,
                         int rowsA, int colsA, int colsB) {

    // Calculate row index
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Calculate column index
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check
    if (row < rowsA && col < colsB) {

        int sum = 0;

        // Matrix multiplication logic
        for (int i = 0; i < colsA; i++) {

            sum += A[row * colsA + i] *
                   B[i * colsB + col];
        }

        // Store result
        C[row * colsB + col] = sum;
    }
}


// ================= PRINT MATRIX =================

// Function to print matrix
void printMatrix(int* matrix, int rows, int cols) {

    for (int i = 0; i < rows; i++) {

        for (int j = 0; j < cols; j++) {

            cout << matrix[i * cols + j] << " ";
        }

        cout << endl;
    }

    cout << endl;
}


// ================= SAFE INTEGER INPUT =================

// Function for valid integer input
int getInt(string message) {

    int value;

    while (true) {

        cout << message;

        cin >> value;

        // Invalid input check
        if (cin.fail()) {

            cout << "Invalid input! Enter a number.\n";

            cin.clear();

            cin.ignore(numeric_limits<streamsize>::max(), '\n');
        }

        // Positive number check
        else if (value <= 0) {

            cout << "Value must be greater than 0.\n";
        }

        else {

            return value;
        }
    }
}


// ================= MATRIX INPUT =================

// Function to input matrix values
void inputMatrix(int* mat,
                 int rows,
                 int cols,
                 string name) {

    cout << "\nEnter elements of Matrix "
         << name << ":\n";

    for (int i = 0; i < rows; i++) {

        for (int j = 0; j < cols; j++) {

            while (true) {

                cout << name
                     << "[" << i << "]["
                     << j << "] = ";

                cin >> mat[i * cols + j];

                // Invalid input check
                if (cin.fail()) {

                    cout << "Invalid input! Enter a number.\n";

                    cin.clear();

                    cin.ignore(
                    numeric_limits<streamsize>::max(),
                    '\n');
                }

                else {

                    break;
                }
            }
        }
    }
}


int main() {

    int rowsA, colsA;
    int rowsB, colsB;

    // ================= MATRIX SIZE INPUT =================

    rowsA = getInt("Enter rows of Matrix A: ");

    colsA = getInt("Enter cols of Matrix A: ");


    // Validate matrix multiplication condition
    while (true) {

        rowsB = getInt("Enter rows of Matrix B: ");

        colsB = getInt("Enter cols of Matrix B: ");

        // cols(A) must equal rows(B)
        if (colsA != rowsB) {

            cout << "\nInvalid! cols(A) must equal rows(B).\n";

            cout << "Try again.\n\n";
        }

        else {

            break;
        }
    }


    // ================= HOST MEMORY =================

    // CPU memory allocation
    int* A = new int[rowsA * colsA];

    int* B = new int[rowsB * colsB];

    int* C = new int[rowsA * colsB];


    // ================= INPUT MATRICES =================

    inputMatrix(A, rowsA, colsA, "A");

    inputMatrix(B, rowsB, colsB, "B");


    // Print matrices
    cout << "\nMatrix A:\n";

    printMatrix(A, rowsA, colsA);

    cout << "Matrix B:\n";

    printMatrix(B, rowsB, colsB);


    // ================= DEVICE MEMORY =================

    // GPU pointers
    int *d_A, *d_B, *d_C;

    // Allocate GPU memory
    cudaMalloc(&d_A,
               rowsA * colsA * sizeof(int));

    cudaMalloc(&d_B,
               rowsB * colsB * sizeof(int));

    cudaMalloc(&d_C,
               rowsA * colsB * sizeof(int));


    // ================= COPY HOST -> DEVICE =================

    cudaMemcpy(d_A, A,
               rowsA * colsA * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B,
               rowsB * colsB * sizeof(int),
               cudaMemcpyHostToDevice);


    // ================= THREADS AND BLOCKS =================

    int THREADS = 16;

    // 2D thread structure
    dim3 threads(THREADS, THREADS);

    // 2D block structure
    dim3 blocks(

        (colsB + THREADS - 1) / THREADS,

        (rowsA + THREADS - 1) / THREADS
    );


    // ================= KERNEL LAUNCH =================

    multiply<<<blocks, threads>>>(
        d_A, d_B, d_C,
        rowsA, colsA, colsB
    );


    // Wait until GPU completes
    cudaDeviceSynchronize();


    // ================= COPY DEVICE -> HOST =================

    cudaMemcpy(C, d_C,
               rowsA * colsB * sizeof(int),
               cudaMemcpyDeviceToHost);


    // ================= PRINT RESULT =================

    cout << "Result Matrix:\n";

    printMatrix(C, rowsA, colsB);


    // ================= CLEANUP =================

    // Free CPU memory
    delete[] A;
    delete[] B;
    delete[] C;

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matmul.cu


In [ ]:
!nvcc matmul.cu -o matmul
!./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter rows of Matrix A: 2
Enter cols of Matrix A: 3
Enter rows of Matrix B: 3
Enter cols of Matrix B: 2

Enter elements of Matrix A:
A[0][0] = 2
A[0][1] = 6
A[0][2] = 7
A[1][0] = 3
A[1][1] = 9
A[1][2] = 4

Enter elements of Matrix B:
B[0][0] = 7
B[0][1] = 3
B[1][0] = 1
B[1][1] = 8
B[2][0] = 4
B[2][1] = 8

Matrix A:
2 6 7 
3 9 4 

Matrix B:
7 3 
1 8 
4 8 

Result Matrix:
48 110 
46 113 

